# Agentic RAG — LangGraph + LangChain + Qdrant + OpenAI

A plain RAG bot only retrieves and answers. In this project the agent **makes decisions**:

1. It checks whether the retrieved documents are actually relevant to the question (`grade_documents`).
2. If no relevant document is found, it searches the web instead (`web_search`, via Tavily).
3. After writing an answer, it checks whether the answer is grounded in the documents and whether it actually resolves the question (`grade_generation`). If not, it rewrites the answer or falls back to a web search.

**Stack:** LangGraph (decision graph) · LangChain (plumbing) · Qdrant (vector DB, embedded — no server needed) · OpenAI (`gpt-4o-mini` + `text-embedding-3-small`)

This notebook runs in **Google Colab** as well as local Jupyter.


## 1) Install libraries

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters \
    langchain-qdrant langgraph qdrant-client tavily-python pymupdf python-dotenv pydantic


## 2) Enter API keys

We use `getpass` to enter keys safely — nothing is echoed to the screen and nothing is
saved inside the notebook file. This works the same way in Colab and local Jupyter, and
avoids the Colab Secrets "Notebook access" toggle issues.

- **OPENAI_API_KEY** — required. Get one at https://platform.openai.com/api-keys
- **TAVILY_API_KEY** — needed for the web-search fallback. Free tier: https://tavily.com
  (1000 requests/month free). If you skip it, the web-search step will simply do nothing —
  the agent will still work using only your documents.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass(
        "TAVILY_API_KEY (optional, press Enter to skip): "
    )

print("Keys set.")


## 3) Configuration

Models, embeddings, and the Qdrant vector store are created once here. Qdrant runs
**embedded** (`QdrantClient(path=...)`) — no separate server needed, data is written to
disk at `./qdrant_db`.

> **Watch out:** `text-embedding-3-small` produces 1536-dimensional vectors. If you switch
> to a different embedding model later, change `QDRANT_COLLECTION`'s name — otherwise the
> dimensions won't match and you'll get an error.


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
VISION_MODEL = "gpt-4o-mini"
EMBEDDING_DIM = 1536

QDRANT_PATH = "./qdrant_db"
QDRANT_COLLECTION = "agentic_rag_openai"

MAX_RETRIES = 3  # how many times the "generate" node may re-run (guards against infinite loops)


def get_llm(temperature: float = 0.0) -> ChatOpenAI:
    return ChatOpenAI(model=CHAT_MODEL, temperature=temperature)


def get_vision_llm() -> ChatOpenAI:
    return ChatOpenAI(model=VISION_MODEL, temperature=0.0)


def get_embeddings() -> OpenAIEmbeddings:
    return OpenAIEmbeddings(model=EMBEDDING_MODEL)


_qdrant_client = QdrantClient(path=QDRANT_PATH)

_existing = [c.name for c in _qdrant_client.get_collections().collections]
if QDRANT_COLLECTION not in _existing:
    _qdrant_client.create_collection(
        collection_name=QDRANT_COLLECTION,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )

vectorstore = QdrantVectorStore(
    client=_qdrant_client,
    collection_name=QDRANT_COLLECTION,
    embedding=get_embeddings(),
)

print("Qdrant ready. Collection:", QDRANT_COLLECTION)


## 4) Ingest documents — multimodal

Both **text and images** inside a PDF become searchable:
- Text is split into chunks directly.
- Each image is sent to the vision model (`gpt-4o-mini`), captioned, and added as its own
  "document" — this way diagrams and charts are searchable just like text.

The cell after this one uses `files.upload()` to upload files in Colab (if you're running
locally instead, just point `run_ingest(...)` at a file path directly).


In [ ]:
import base64
from pathlib import Path

import fitz  # PyMuPDF
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150


def caption_image(image_bytes: bytes, vision_llm) -> str:
    b64 = base64.b64encode(image_bytes).decode("utf-8")
    msg = HumanMessage(
        content=[
            {
                "type": "text",
                "text": (
                    "Describe this image/diagram in 1-3 sentences, focusing on any "
                    "text, labels, numbers, or technical content visible in it. "
                    "Be factual and specific."
                ),
            },
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
        ]
    )
    return vision_llm.invoke([msg]).content


def load_pdf(path: Path, vision_llm) -> list:
    docs = []
    pdf = fitz.open(path)

    for page_num, page in enumerate(pdf, start=1):
        text = page.get_text().strip()
        if text:
            docs.append(Document(page_content=text, metadata={"source": path.name, "page": page_num, "type": "text"}))

        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            base_image = pdf.extract_image(xref)
            image_bytes = base_image["image"]
            if len(image_bytes) < 3000:  # skip tiny icons
                continue
            try:
                caption = caption_image(image_bytes, vision_llm)
            except Exception as e:
                print(f"  [!] Failed to caption image ({path.name} p{page_num}): {e}")
                continue
            docs.append(Document(
                page_content=f"[Image] {caption}",
                metadata={"source": path.name, "page": page_num, "type": "image", "image_index": img_index},
            ))
            print(f"  - p{page_num} image #{img_index} captioned")

    pdf.close()
    return docs


def load_text_file(path: Path) -> list:
    text = path.read_text(encoding="utf-8", errors="ignore")
    return [Document(page_content=text, metadata={"source": path.name, "type": "text"})]


def load_path(path: Path, vision_llm) -> list:
    docs = []
    files = [path] if path.is_file() else sorted(path.rglob("*"))
    for f in files:
        if not f.is_file():
            continue
        if f.suffix.lower() == ".pdf":
            print(f"[PDF] reading {f.name}...")
            docs.extend(load_pdf(f, vision_llm))
        elif f.suffix.lower() in {".txt", ".md"}:
            print(f"[TXT] reading {f.name}...")
            docs.extend(load_text_file(f))
        else:
            print(f"[!] Skipped (unsupported): {f.name}")
    return docs


def run_ingest(source_path: str):
    path = Path(source_path)
    if not path.exists():
        raise FileNotFoundError(f"Not found: {source_path}")

    vision_llm = get_vision_llm()
    raw_docs = load_path(path, vision_llm)
    if not raw_docs:
        print("No documents found.")
        return

    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(raw_docs)
    print(f"\n{len(raw_docs)} document(s) -> {len(chunks)} chunk(s).")

    vectorstore.add_documents(chunks)
    print(f"Written to Qdrant successfully. ({len(chunks)} chunks)")


In [ ]:
# --- Upload files in Colab ---
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    run_ingest(fname)

# --- Or, if running locally, point directly at a folder/file path instead ---
# run_ingest("data/my_document.pdf")


## 5) Graph state

The shared state that "flows" through the LangGraph graph. Every node reads this state
and returns updates to it.


In [ ]:
from typing import List, TypedDict


class GraphState(TypedDict, total=False):
    question: str            # the user's question
    generation: str          # the answer the LLM wrote
    documents: List[str]     # text chunks currently in context
    sources: List[dict]      # for citations: {"source":.., "page":.., "type":..}
    web_used: bool            # whether the web fallback was used
    retries: int               # how many times "generate" has re-run
    steps: List[str]            # sequence of nodes visited (for debug/UI)


## 6) Nodes

Every "circle/diamond" of the graph lives here. The graders (`relevance`, `groundedness`,
`answers_question`) get a reliable **yes/no** from the LLM via `with_structured_output` —
not free-form text — which is what makes the routing stable.


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.tavily_search import TavilySearchResults


class RelevanceGrade(BaseModel):
    '''Is a single document chunk relevant to the question?'''
    binary_score: str = Field(description="'yes' or 'no'")


class GroundednessGrade(BaseModel):
    '''Is the answer grounded in / supported by the given documents (no hallucination)?'''
    binary_score: str = Field(description="'yes' - grounded, 'no' - made up")


class AnswersQuestionGrade(BaseModel):
    '''Does the answer actually resolve the question?'''
    binary_score: str = Field(description="'yes' or 'no'")


In [ ]:
# --- 1) RETRIEVE ---

def retrieve(state: GraphState) -> dict:
    question = state["question"]
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    docs = retriever.invoke(question)

    return {
        "documents": [d.page_content for d in docs],
        "sources": [
            {"source": d.metadata.get("source", "unknown"), "page": d.metadata.get("page"), "type": d.metadata.get("type", "text")}
            for d in docs
        ],
        "web_used": False,
        "retries": 0,
        "steps": ["retrieve"],
    }


In [ ]:
# --- 2) GRADE DOCUMENTS ---

_relevance_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a grader assessing relevance of a retrieved document chunk to a "
               "user question. Give a binary score 'yes' or 'no'. 'yes' means the chunk "
               "contains keywords or semantic meaning related to the question, even partially."),
    ("human", "Retrieved chunk:\n\n{document}\n\nUser question: {question}"),
])


def grade_documents(state: GraphState) -> dict:
    llm = get_llm()
    grader = _relevance_prompt | llm.with_structured_output(RelevanceGrade)

    question = state["question"]
    docs = state["documents"]
    sources = state.get("sources", [])

    kept_docs, kept_sources = [], []
    for doc, src in zip(docs, sources):
        result = grader.invoke({"document": doc, "question": question})
        if result.binary_score.strip().lower().startswith("y"):
            kept_docs.append(doc)
            kept_sources.append(src)

    return {
        "documents": kept_docs,
        "sources": kept_sources,
        "steps": state.get("steps", []) + ["grade_documents"],
    }


def route_after_grade(state: GraphState) -> str:
    '''If at least 1 relevant document remains -> generate the answer directly.
    Otherwise (everything got filtered out) -> web search.'''
    if len(state.get("documents", [])) == 0:
        return "web_search"
    return "generate"


In [ ]:
# --- 3) WEB SEARCH (Tavily fallback) ---

def web_search(state: GraphState) -> dict:
    question = state["question"]
    tool = TavilySearchResults(max_results=4)
    items = tool.invoke({"query": question})  # list of {"content":.., "url":..}

    new_docs = [item.get("content", "") for item in items]
    new_sources = [{"source": item.get("url", "web"), "page": None, "type": "web"} for item in items]

    return {
        "documents": state.get("documents", []) + new_docs,
        "sources": state.get("sources", []) + new_sources,
        "web_used": True,
        "steps": state.get("steps", []) + ["web_search"],
    }


In [ ]:
# --- 4) GENERATE ---

_generate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant answering questions using ONLY the provided context. "
               "If the context does not contain the answer, say you don't know — do not "
               "make anything up. Answer in the same language as the question. Be concise "
               "and cite which piece of context you used when relevant."),
    ("human", "Context:\n\n{context}\n\nQuestion: {question}"),
])


def generate(state: GraphState) -> dict:
    llm = get_llm(temperature=0.2)
    chain = _generate_prompt | llm

    context = "\n\n---\n\n".join(state.get("documents", []))
    question = state["question"]

    response = chain.invoke({"context": context, "question": question})

    return {
        "generation": response.content,
        "retries": state.get("retries", 0) + 1,
        "steps": state.get("steps", []) + ["generate"],
    }


In [ ]:
# --- 5) GRADE GENERATION (groundedness + answers-question) ---

_groundedness_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a grader assessing whether an answer is grounded in / supported by "
               "a given set of facts. Give a binary score 'yes' or 'no'. 'yes' means the "
               "answer is supported by the facts, with no fabricated claims."),
    ("human", "Facts:\n\n{documents}\n\nAnswer: {generation}"),
])

_answers_question_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a grader assessing whether an answer actually resolves a user "
               "question. Give a binary score 'yes' or 'no'."),
    ("human", "Question: {question}\n\nAnswer: {generation}"),
])


def route_after_generate(state: GraphState) -> str:
    if state.get("retries", 0) >= MAX_RETRIES:
        return "useful"  # retry cap reached -> avoid an infinite loop

    llm = get_llm()
    documents = "\n\n".join(state.get("documents", []))
    generation = state["generation"]
    question = state["question"]

    grounded = (
        (_groundedness_prompt | llm.with_structured_output(GroundednessGrade))
        .invoke({"documents": documents, "generation": generation})
        .binary_score.strip().lower().startswith("y")
    )
    if not grounded:
        return "not_grounded"

    answers = (
        (_answers_question_prompt | llm.with_structured_output(AnswersQuestionGrade))
        .invoke({"question": question, "generation": generation})
        .binary_score.strip().lower().startswith("y")
    )
    if not answers:
        return "not_useful"

    return "useful"


## 7) Assemble the graph (LangGraph)

This is the heart of the project: nodes + conditional edges build the decision graph.


In [ ]:
from langgraph.graph import StateGraph, END

g = StateGraph(GraphState)
g.add_node("retrieve", retrieve)
g.add_node("grade_documents", grade_documents)
g.add_node("web_search", web_search)
g.add_node("generate", generate)

g.set_entry_point("retrieve")
g.add_edge("retrieve", "grade_documents")

g.add_conditional_edges(
    "grade_documents", route_after_grade,
    {"web_search": "web_search", "generate": "generate"},
)
g.add_edge("web_search", "generate")

g.add_conditional_edges(
    "generate", route_after_generate,
    {"useful": END, "not_grounded": "generate", "not_useful": "web_search"},
)

app = g.compile()
print("Graph ready.")


In [ ]:
# View the graph as an image (optional)
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Visualization skipped:", e)


## 8) Chat with the agent

After every answer, the steps it took and the sources it used are printed — this lets you
see exactly how the agent decided (for example:
`retrieve -> grade_documents -> web_search -> generate`).


In [ ]:
def ask(question: str):
    result = app.invoke({"question": question})

    print("=" * 60)
    print("ANSWER:")
    print(result.get("generation", "(no answer)"))

    print("\nSTEPS TAKEN:", " -> ".join(result.get("steps", [])))

    sources = result.get("sources", [])
    if sources:
        print("\nSOURCES:")
        for s in sources:
            page = f", page {s['page']}" if s.get("page") else ""
            print(f"  - [{s.get('type', 'text')}] {s.get('source')}{page}")
    print("=" * 60)
    return result


In [ ]:
# 1) A question the documents should be able to answer
ask("What is the main topic of the document?")


In [ ]:
# 2) A question the documents likely don't cover -> web_search should kick in
# (only if you entered a TAVILY_API_KEY)
ask("What is today's weather in Tashkent?")


## 9) Next steps (optional)

- **Evaluation**: build 10-15 question/answer pairs and compute retrieval hit rate,
  groundedness, and answer relevance metrics.
- **Experiments**: compare `CHUNK_SIZE` (500/1000/2000) and `k` (2/4/8); measure the
  difference with/without `grade_documents`.
- **Gradio interface**: wrap `ask()` with Gradio to get a public shareable link (below).


In [ ]:
!pip install -q gradio


In [ ]:
import gradio as gr


def gradio_ask(question, history):
    result = app.invoke({"question": question})
    answer = result.get("generation", "(no answer)")

    steps = " -> ".join(result.get("steps", []))
    sources = result.get("sources", [])
    src_lines = []
    for s in sources:
        page = f", page {s['page']}" if s.get("page") else ""
        src_lines.append(f"- [{s.get('type', 'text')}] {s.get('source')}{page}")

    footer = f"\n\n---\n**Steps:** {steps}"
    if src_lines:
        footer += "\n**Sources:**\n" + "\n".join(src_lines)

    return answer + footer


demo = gr.ChatInterface(
    fn=gradio_ask,
    title="Agentic RAG",
    description="Ask a question based on your documents. The agent searches the web if "
                 "needed and checks its own answer before replying.",
    examples=["What is the main topic of the document?"],
)

demo.launch(share=True, debug=False)
